# Publication-safe notebook

This generalized reference implementation expects a live SparkSession named
`spark` and pseudonymous tables matching `docs/data-contract.md`. It contains no
institution-specific ingestion, credentials, internal paths, data, or results.

Parameter defaults are illustrative and intentionally differ from internal
operating settings. Calibrate them only with authorized development data.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import StorageLevel

import os

WRITE_DB = os.environ.get("FRAUD_GRAPH_WRITE_DB", "fraud_graph_demo")
EMP = os.environ.get("FRAUD_GRAPH_PREFIX", "onboarding_fraud")
EDGES_TXN = f"{WRITE_DB}.{EMP}_v2_edges_transferred"
NODES_ACCT = f"{WRITE_DB}.{EMP}_v2_nodes_account"
SUBG_NIDX = f"{WRITE_DB}.{EMP}_v2_subgraph_node_index"
OWN_TBL  = f"{WRITE_DB}.{EMP}_v2_edges_ownership"

ANCHOR_TRAILING = {
    "202401": ["202310", "202311", "202312"],
    "202402": ["202311", "202312", "202401"],
    "202403": ["202312", "202401", "202402"],
}

TEST_ONE_ANCHOR = False

RUN_COMMUNITIES = False
ALPHA = float(os.environ.get("FRAUD_GRAPH_PPR_ALPHA", "0.80"))
PPR_ITERS = int(os.environ.get("FRAUD_GRAPH_PPR_ITERS", "10"))
LPA_ITERS = int(os.environ.get("FRAUD_GRAPH_LPA_ITERS", "5"))
CC_MAX_ITERS = int(os.environ.get("FRAUD_GRAPH_CC_MAX_ITERS", "20"))
KHOP_EXPOSURE = int(os.environ.get("FRAUD_GRAPH_EXPOSURE_HOPS", "2"))
K_PCTS = [float(value) for value in os.environ.get("FRAUD_GRAPH_EVALUATION_PCTS", "0.02,0.05").split(",")]
CKPT_EVERY = int(os.environ.get("FRAUD_GRAPH_CHECKPOINT_EVERY", "5"))

RARE_BENEF_MAX = int(os.environ.get("FRAUD_GRAPH_RARE_BENEF_MAX", "25"))

OUT_FEATS = f"{WRITE_DB}.{EMP}_v2_node_features"
OUT_CUST  = f"{WRITE_DB}.{EMP}_v2_cust_features"
OUT_LIFT  = f"{WRITE_DB}.{EMP}_v2_increment_lift"

def save_overwrite(df, t):
    spark.sql("DROP TABLE IF EXISTS " + t); df.write.mode("overwrite").saveAsTable(t)

anchors = (["202401"] if TEST_ONE_ANCHOR else list(ANCHOR_TRAILING.keys()))
print("anchors:", anchors, "| communities:", RUN_COMMUNITIES, "| PPR iters:", PPR_ITERS, "| alpha:", ALPHA)


In [ ]:
USE_GRAPHFRAMES = True
GF_JAR = os.environ.get("GRAPHFRAMES_JAR", "graphframes.jar")
CKPT_DIR = os.environ.get("FRAUD_GRAPH_CHECKPOINT_DIR", "file:///tmp/onboarding_fraud/gf_ckpt")
GraphFrame = None

spark.sparkContext.setCheckpointDir(CKPT_DIR)
print("checkpoint directory:", CKPT_DIR)
if USE_GRAPHFRAMES:
    try:
        spark.sparkContext.addPyFile(GF_JAR)
        from graphframes import GraphFrame
        print("GraphFrames enabled.")
    except Exception as ge:
        GraphFrame = None
        print("GraphFrames import failed (only needed if RUN_COMMUNITIES=True):", ge)
else:
    print("GraphFrames disabled — using the Spark fallback implementation.")


In [ ]:
def build_substrate(anchor):
    months = ANCHOR_TRAILING[anchor]
    e0 = (spark.table(EDGES_TXN).where(F.col("txn_month").isin(*months))
          .select("src", "dst", F.col("sum_amount").cast("double").alias("amt"),
                  F.col("txn_month").cast("int").alias("tm"))
          .where(F.col("src") != F.col("dst")).persist(StorageLevel.DISK_ONLY))

    fanin = e0.groupBy("dst").agg(F.countDistinct("src").alias("fi"))
    rare = fanin.where(F.col("fi") <= RARE_BENEF_MAX).select("dst")
    e = (e0.join(rare, "dst", "inner")
         .groupBy("src", "dst").agg(F.sum("amt").alias("amt"))
         .withColumn("weight", F.log1p(F.col("amt")))
         .select("src", "dst", "weight").persist(StorageLevel.DISK_ONLY))

    de = (e0.join(rare, "dst", "inner").select("src", "dst", "tm").distinct().persist(StorageLevel.DISK_ONLY))
    e.count(); de.count(); e0.unpersist()

    und = (e.select("src", "dst", "weight")
           .union(e.select(F.col("dst").alias("src"), F.col("src").alias("dst"), "weight"))
           .persist(StorageLevel.DISK_ONLY))
    lab = spark.table(SUBG_NIDX).where(F.col("anchor_month") == anchor)
    seeds = lab.where(F.col("is_known_bad") == 1).select(F.col("id")).distinct()

    cands = (lab.where((F.col("is_candidate") == 1) & (F.col("is_known_bad") == 0))
             .select(F.col("id"), F.col("target_fraud").cast("int").alias("target_fraud")))
    return e, und, de, seeds, cands


In [ ]:
def feat_degree(e):
    out_d = e.groupBy("src").agg(F.sum("weight").alias("out_w"), F.count("*").alias("out_deg"))\
             .select(F.col("src").alias("id"), "out_w", "out_deg")
    in_d  = e.groupBy("dst").agg(F.sum("weight").alias("in_w"), F.count("*").alias("in_deg"))\
             .select(F.col("dst").alias("id"), "in_w", "in_deg")
    return out_d.join(in_d, "id", "outer").na.fill(0.0)


In [ ]:
def _temporal_reach(de, seeds, khop, forward):

    frontier = seeds.select("id").withColumn("b", F.lit(0) if forward else F.lit(999999))
    visited  = seeds.select("id").withColumn("hop", F.lit(0))
    for h in range(1, khop + 1):
        if forward:
            nbr = (de.alias("e").join(frontier.alias("f"), F.col("e.src") == F.col("f.id"))
                   .where(F.col("e.tm") >= F.col("f.b"))
                   .select(F.col("e.dst").alias("id"), F.col("e.tm").alias("b")))
            nbr = nbr.groupBy("id").agg(F.min("b").alias("b"))
        else:
            nbr = (de.alias("e").join(frontier.alias("f"), F.col("e.dst") == F.col("f.id"))
                   .where(F.col("e.tm") <= F.col("f.b"))
                   .select(F.col("e.src").alias("id"), F.col("e.tm").alias("b")))
            nbr = nbr.groupBy("id").agg(F.max("b").alias("b"))
        new = nbr.join(visited.select("id"), "id", "left_anti")
        new = new.checkpoint(eager=True)
        if new.limit(1).count() == 0:
            break
        visited  = visited.unionByName(new.select("id").withColumn("hop", F.lit(h)))
        frontier = new.select("id", "b")
    return visited.select("id", "hop")

def feat_seed_exposure(de, seeds, khop):

    fwd = _temporal_reach(de, seeds, khop, True)
    bwd = _temporal_reach(de, seeds, khop, False)
    mh = fwd.unionByName(bwd).groupBy("id").agg(F.min("hop").alias("min_hops_to_bad"))
    return mh.withColumn("closeness_to_bad", 1.0 / (1.0 + F.col("min_hops_to_bad")))


In [ ]:
def feat_shared_seed_benef(e, seeds):

    seed_benef = (e.join(seeds.select(F.col("id").alias("src")), "src")
                  .select("dst").distinct())

    return (e.join(seed_benef, "dst")
            .groupBy("src")
            .agg(F.countDistinct("dst").alias("n_shared_mule"))
            .select(F.col("src").alias("id"), "n_shared_mule",
                    F.lit(1).alias("shares_mule")))


In [ ]:
def feat_ppr(und, seeds, alpha, iters):
    deg = und.groupBy("src").agg(F.sum("weight").alias("wsum")).select(F.col("src").alias("id"), "wsum")
    n = seeds.count()
    reset = seeds.select(F.col("id"), F.lit(1.0 / n).alias("reset")).persist(StorageLevel.DISK_ONLY)
    score = reset.select("id", F.col("reset").alias("score"))
    for it in range(iters):
        contrib = (und.alias("u")
                   .join(score.alias("s"), F.col("u.src") == F.col("s.id"))
                   .join(deg.alias("d"), F.col("u.src") == F.col("d.id"))
                   .select(F.col("u.dst").alias("id"),
                           (F.col("s.score") * F.col("u.weight") / F.col("d.wsum")).alias("c")))
        inc = contrib.groupBy("id").agg(F.sum("c").alias("inc"))
        score = (inc.join(reset, "id", "full_outer").na.fill(0.0)
                 .select("id", (F.lit(alpha) * F.col("inc") + F.lit(1 - alpha) * F.col("reset")).alias("score")))

        if (it + 1) % CKPT_EVERY == 0:
            score = score.checkpoint(eager=True)
    return score.select("id", F.col("score").alias("ppr"))


In [ ]:
def connected_components(e):

    ee = (e.select(F.least("src", "dst").alias("src"), F.greatest("src", "dst").alias("dst"))
          .where(F.col("src") != F.col("dst")).distinct().persist(StorageLevel.DISK_ONLY))

    def min_nbrs(x):
        sym = x.select("src", "dst").union(x.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
        return (sym.groupBy("src").agg(F.min("dst").alias("min_nbr"))
                .withColumn("min_nbr", F.least("src", "min_nbr")))

    prev = None
    for it in range(CC_MAX_ITERS):
        m1 = min_nbrs(ee)
        ee = (ee.alias("e").join(m1.alias("m"), F.col("e.src") == F.col("m.src"))
              .select(F.col("e.dst").alias("src"), F.col("m.min_nbr").alias("dst"))
              .where(F.col("src") != F.col("dst")).distinct())
        m2 = ee.groupBy("src").agg(F.min("dst").alias("min_nbr"))
        ee = (ee.alias("e").join(m2.alias("m"), "src")
              .select(F.col("m.min_nbr").alias("src"), F.col("e.dst").alias("dst"))
              .where(F.col("src") != F.col("dst"))
              .union(m2.select(F.col("min_nbr").alias("src"), F.col("src").alias("dst")))
              .distinct())
        ee = ee.localCheckpoint(eager=True)
        s = min_nbrs(ee).agg(F.sum(F.hash("min_nbr").cast("double"))).first()[0]
        if s == prev:
            break
        prev = s

    comp = ee.select(F.col("dst").alias("id"), F.col("src").alias("component"))

    allnodes = e.select(F.col("src").alias("id")).union(e.select(F.col("dst").alias("id"))).distinct()
    comp = (allnodes.join(comp, "id", "left")
            .select("id", F.coalesce(F.col("component"), F.col("id")).alias("component")))
    return comp


In [ ]:
def label_propagation(und, allnodes, iters):
    lab = allnodes.select("id", F.col("id").alias("label"))
    for it in range(iters):
        msgs = (und.alias("u").join(lab.alias("l"), F.col("u.src") == F.col("l.id"))
                .select(F.col("u.dst").alias("id"), F.col("l.label")))
        cnt = msgs.groupBy("id", "label").count()
        w = Window.partitionBy("id").orderBy(F.desc("count"), F.asc("label"))
        new = cnt.withColumn("rn", F.row_number().over(w)).where(F.col("rn") == 1).select("id", "label")
        new = new.localCheckpoint(eager=True)
        changed = (new.alias("n").join(lab.alias("o"), "id")
                   .where(F.col("n.label") != F.col("o.label")).limit(1).count())
        lab = new
        if changed == 0:
            break
    return lab.select("id", F.col("label").alias("lpa_comm"))

def community_bad_rate(comm, seeds, comm_col):

    sc  = comm.join(seeds.select("id"), "id", "inner").groupBy(comm_col).agg(F.count("*").alias("n_seed"))
    tot = comm.groupBy(comm_col).agg(F.count("*").alias("n_total"))
    rate = (tot.join(sc, comm_col, "left").na.fill(0, ["n_seed"])
            .withColumn("comm_bad_rate", F.col("n_seed") / F.col("n_total")).select(comm_col, "comm_bad_rate"))
    return comm.join(rate, comm_col, "left").na.fill(0.0, ["comm_bad_rate"]).select("id", comm_col, "comm_bad_rate")


In [ ]:
def capture_at(df, score_col, ks):
    w = Window.orderBy(F.col(score_col).desc_nulls_last())
    ranked = df.select("target_fraud", F.row_number().over(w).alias("rk"))
    tot = df.agg(F.count("*").alias("n"), F.sum("target_fraud").alias("nbad")).first()
    n, nbad = tot["n"], (tot["nbad"] or 0)
    rows = []
    for p in ks:
        K = max(1, int(n * p))
        topbad = ranked.where((F.col("rk") <= K) & (F.col("target_fraud") == 1)).count()
        cap = (topbad / nbad) if nbad else 0.0
        lift = ((topbad / K) / (nbad / n)) if nbad else 0.0
        rows.append((p, K, topbad, round(cap, 4), round(lift, 3)))
    return rows, n, nbad

def pr(df, col):
    w = Window.orderBy(F.col(col).asc_nulls_first())
    return df.withColumn("pr_" + col, F.percent_rank().over(w))


In [ ]:
lift_rows = []
FEAT_COLS = ["id", "anchor_month", "target_fraud", "in_deg", "out_deg", "in_w", "out_w",
             "min_hops_to_bad", "closeness_to_bad", "ppr", "n_shared_mule", "shares_mule", "comm_bad_rate"]
INCREMENTS = [("baseline", ["pr_in_deg", "pr_out_deg"]),
              ("+multihop", ["pr_closeness_to_bad"]),
              ("+ppr",      ["pr_ppr"]),
              ("+sharedmule", ["pr_n_shared_mule"]),
              ("+community", ["pr_comm_bad_rate"])]

first_save = True
first_cust = True
for a in anchors:
    print("=== anchor", a, "===")
    e, und, de, seeds, cands = build_substrate(a)

    deg = feat_degree(e)
    expo = feat_seed_exposure(de, seeds, KHOP_EXPOSURE)
    ppr = feat_ppr(und, seeds, ALPHA, PPR_ITERS)
    shared = feat_shared_seed_benef(e, seeds)
    feats = (cands.join(deg, "id", "left").join(expo, "id", "left")
             .join(ppr, "id", "left").join(shared, "id", "left"))

    if RUN_COMMUNITIES:
        try:
            allnodes = e.select(F.col("src").alias("id")).union(e.select(F.col("dst").alias("id"))).distinct()
            if USE_GRAPHFRAMES:
                g = GraphFrame(allnodes.select("id"), und.select("src", "dst"))
                comm = g.labelPropagation(maxIter=LPA_ITERS).select("id", F.col("label").alias("comm"))
            else:
                comm = label_propagation(und, allnodes, LPA_ITERS).select("id", F.col("lpa_comm").alias("comm"))
            cbr = community_bad_rate(comm, seeds, "comm")
            feats = feats.join(cbr.select("id", "comm_bad_rate"), "id", "left")
        except Exception as ce:
            print("   [WARN] community step failed — continuing with comm_bad_rate=0:", ce)
            feats = feats.withColumn("comm_bad_rate", F.lit(0.0))
    else:
        feats = feats.withColumn("comm_bad_rate", F.lit(0.0))

    feats = feats.na.fill({"in_deg": 0, "out_deg": 0, "in_w": 0.0, "out_w": 0.0,
                           "min_hops_to_bad": 99, "closeness_to_bad": 0.0, "ppr": 0.0,
                           "n_shared_mule": 0, "shares_mule": 0, "comm_bad_rate": 0.0})
    feats = feats.withColumn("anchor_month", F.lit(a)).persist(StorageLevel.DISK_ONLY)

    out = feats.select(*FEAT_COLS)
    if first_save:
        save_overwrite(out, OUT_FEATS); first_save = False
    else:
        out.write.mode("append").saveAsTable(OUT_FEATS)
    print("   saved account features for", a, "->", OUT_FEATS)

    own = spark.table(OWN_TBL).select(F.col("dst").alias("id"), F.col("src").alias("cust"))
    cust = (feats.join(own, "id", "inner").groupBy("cust").agg(
                F.max("in_deg").alias("in_deg"), F.max("out_deg").alias("out_deg"),
                F.max("in_w").alias("in_w"), F.max("out_w").alias("out_w"),
                F.min("min_hops_to_bad").alias("min_hops_to_bad"),
                F.max("closeness_to_bad").alias("closeness_to_bad"),
                F.max("ppr").alias("ppr"),
                F.max("n_shared_mule").alias("n_shared_mule"), F.max("shares_mule").alias("shares_mule"),
                F.max("comm_bad_rate").alias("comm_bad_rate"),
                F.max("target_fraud").alias("target_fraud"))
            .withColumn("anchor_month", F.lit(a)).persist(StorageLevel.DISK_ONLY))
    cout = cust.select("cust", "anchor_month", "target_fraud", "in_deg", "out_deg", "in_w", "out_w",
                       "min_hops_to_bad", "closeness_to_bad", "ppr", "n_shared_mule", "shares_mule", "comm_bad_rate")
    if first_cust:
        save_overwrite(cout, OUT_CUST); first_cust = False
    else:
        cout.write.mode("append").saveAsTable(OUT_CUST)
    print("   saved customer rollup for", a, "->", OUT_CUST)

    for c in ["in_deg", "out_deg", "closeness_to_bad", "ppr", "n_shared_mule", "comm_bad_rate"]:
        cust = pr(cust, c)
    included = []
    for step, cols in INCREMENTS:
        if step == "+community" and not RUN_COMMUNITIES:
            continue
        included = included + cols
        ens = cust.withColumn("ens", sum(F.col(c) for c in included))
        rows, n, nbad = capture_at(ens.select("target_fraud", "ens"), "ens", K_PCTS)
        for (p, K, topbad, cap, lift) in rows:
            lift_rows.append((a, step, p, K, n, nbad, topbad, cap, lift))
            print("   [%s] %-10s top%4.0f%% K=%d  capture=%.3f  lift=%.2f  (bad-cust %d/%d)"
                  % (a, step, p * 100, K, cap, lift, topbad, nbad))

    cust.unpersist(); feats.unpersist(); e.unpersist(); und.unpersist(); de.unpersist()


In [ ]:
print("account features:", OUT_FEATS, "| rows:", spark.table(OUT_FEATS).count())
print("customer rollup :", OUT_CUST, "| rows:", spark.table(OUT_CUST).count())
lift_df = spark.createDataFrame(
    lift_rows, ["anchor_month", "step", "top_pct", "K", "n_cust", "n_bad_cust", "bad_in_topK", "capture", "lift"])
save_overwrite(lift_df, OUT_LIFT)
print("saved incremental-lift log (CUSTOMER level):", OUT_LIFT)
spark.table(OUT_LIFT).orderBy("anchor_month", "step", "top_pct").show(60, False)


In [ ]:
RUN_SLOPING = True
SLOPING_FEATURES = ["n_shared_mule", "shares_mule", "ppr", "closeness_to_bad", "comm_bad_rate", "in_deg", "out_deg"]
OUT_SLOPE = f"{WRITE_DB}.{EMP}_v2_feature_sloping"

def _woe_iv(cnt, bad):
    good = cnt - bad
    tb, tg = bad.sum(), good.sum()
    pb = (bad / tb).clip(lower=1e-6) if tb else bad * 0 + 1e-6
    pg = (good / tg).clip(lower=1e-6) if tg else good * 0 + 1e-6
    woe = np.log(pg / pb)
    return float(((pg - pb) * woe).sum())

def _robust_bins(x, max_bins=10):

    s = pd.Series(np.asarray(x, dtype=float))
    nuniq = int(s.nunique(dropna=True))
    if nuniq <= 1:
        return np.zeros(len(s), dtype=int)
    if nuniq <= max_bins:
        order = {v: i for i, v in enumerate(sorted(s.dropna().unique()))}
        return s.map(order).fillna(0).astype(int).values
    z = (s == 0)
    out = np.zeros(len(s), dtype=int)
    if z.mean() >= 0.02 and int((~z).sum()) >= max_bins:
        pos = ~z
        qr = pd.qcut(s[pos].rank(method="first"),
                     q=min(max_bins - 1, int((~z).sum())), labels=False, duplicates="drop")
        out[pos.values] = 1 + qr.astype(int).values
        return out
    return pd.qcut(s.rank(method="first"), q=max_bins, labels=False, duplicates="drop").astype(int).values

if RUN_SLOPING:
    pdf = spark.table(OUT_CUST).select("anchor_month", "target_fraud", *SLOPING_FEATURES).toPandas()
    slope_rows = []
    for a in sorted(pdf["anchor_month"].unique()):
        d = pdf[pdf["anchor_month"] == a]
        y = d["target_fraud"].astype(int).values
        tot_bad = int(y.sum())
        print("\n=== sloping @ %s (bad=%d / n=%d) ===" % (a, tot_bad, len(d)))
        for f in SLOPING_FEATURES:
            x = d[f].astype(float)
            try:
                b = _robust_bins(x.values, max_bins=10)
                g = pd.DataFrame({"b": b, "y": y})
                agg = g.groupby("b").agg(cnt=("y", "size"), bad=("y", "sum")).sort_index()
                agg["event_rate_pct"] = 100.0 * agg["bad"] / agg["cnt"]
                agg["pct_fraud_captured"] = (100.0 * agg["bad"] / tot_bad) if tot_bad else 0.0
                iv = _woe_iv(agg["cnt"], agg["bad"])
                er = agg["event_rate_pct"].values
                mono = bool(np.all(np.diff(er) >= -1e-9) or np.all(np.diff(er) <= 1e-9))
                print("  %-18s IV=%.3f  bins=%d  monotonic=%s  top-bin event=%.2f%%"
                      % (f, iv, len(agg), mono, er[-1] if len(er) else 0))
                for bi, r in agg.reset_index().iterrows():
                    slope_rows.append((a, f, int(r["b"]) if pd.notna(r["b"]) else -1,
                                       int(r["cnt"]), int(r["bad"]),
                                       round(float(r["event_rate_pct"]), 4),
                                       round(float(r["pct_fraud_captured"]), 4),
                                       round(iv, 4), mono))
            except Exception as fe:
                print("  %-18s sloping skipped: %s" % (f, fe))
    if slope_rows:
        slope_df = spark.createDataFrame(
            slope_rows, ["anchor_month", "feature", "bin", "cnt", "bad",
                         "event_rate_pct", "pct_fraud_captured", "iv", "monotonic"])
        save_overwrite(slope_df, OUT_SLOPE)
        print("\nsaved sloping table:", OUT_SLOPE)


In [ ]:
print("=== Notebook 04 (v2) summary ===")
print("account features rows:", spark.table(OUT_FEATS).count())
cf = spark.table(OUT_CUST)
print("customer rows:", cf.count(), "| anchors:", [r[0] for r in cf.select("anchor_month").distinct().collect()])
cf.groupBy("anchor_month").agg(
    F.count("*").alias("n_cust"),
    F.sum("target_fraud").alias("bad_cust"),
    F.round(F.avg("ppr"), 8).alias("avg_ppr"),
    F.round(F.avg("closeness_to_bad"), 4).alias("avg_closeness")).orderBy("anchor_month").show()
print("NOTEBOOK 04 (v2) COMPLETE — Spark full-graph features (degree, seed-exposure, PPR, community) +")
print("incremental-lift (capture@K) + per-feature binning/sloping (event rate, % captured, IV, monotonicity).")
print("Tables:", OUT_FEATS, "(acct features) |", OUT_CUST, "(customer rollup = eval grain) |")
print("       ", OUT_LIFT, "(lift, customer-level) |", OUT_SLOPE, "(sloping diagnostics, customer-level).")
print("Self-seeds excluded from candidates; lift & sloping are CUSTOMER-level (the example early-warning grain).")
print("Next: 05 (motif features, same incremental gate) -> 06 (select + score + final validation).")
print("If TEST_ONE_ANCHOR=True or RUN_COMMUNITIES=False, paste the lift table, then flip the flags for the full run.")


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import IPython.display as _disp
from pyspark.sql import functions as _F

CUST_TBL      = f"{WRITE_DB}.{EMP}_v2_cust_features"
PLOT_FEATURES = ["n_shared_mule", "shares_mule", "ppr", "closeness_to_bad", "comm_bad_rate", "in_deg", "out_deg"]
PLOT_ANCHOR   = "202401"
MIN_BIN_FRAC = float(os.environ.get("FRAUD_GRAPH_MIN_BIN_FRACTION", "0.02"))
MIN_STABLE_CNT = int(os.environ.get("FRAUD_GRAPH_MIN_STABLE_COUNT", "30"))
MIN_STABLE_BAD = int(os.environ.get("FRAUD_GRAPH_MIN_STABLE_POSITIVES", "3"))
RISKY_ON_LEFT = True

def _wilson_lcb(bad, cnt, z=1.96):
    bad = np.asarray(bad, float); cnt = np.asarray(cnt, float)
    p = np.where(cnt > 0, bad / cnt, 0.0); denom = 1 + z*z/np.maximum(cnt, 1)
    centre = p + z*z/(2*np.maximum(cnt, 1)); margin = z*np.sqrt((p*(1-p) + z*z/(4*np.maximum(cnt,1)))/np.maximum(cnt,1))
    return np.where(cnt > 0, np.clip((centre - margin)/denom, 0, 1), 0.0)

def _robust_bins(x, max_bins=10):
    s = pd.Series(np.asarray(x, dtype=float)); nuniq = int(s.nunique(dropna=True))
    if nuniq <= 1: return np.zeros(len(s), dtype=int)
    if nuniq <= max_bins:
        order = {v: i for i, v in enumerate(sorted(s.dropna().unique()))}
        return s.map(order).fillna(0).astype(int).values
    z = (s == 0); out = np.zeros(len(s), dtype=int)
    if z.mean() >= 0.02 and int((~z).sum()) >= max_bins:
        pos = ~z
        qr = pd.qcut(s[pos].rank(method="first"), q=min(max_bins - 1, int((~z).sum())), labels=False, duplicates="drop")
        out[pos.values] = 1 + qr.astype(int).values; return out
    return pd.qcut(s.rank(method="first"), q=max_bins, labels=False, duplicates="drop").astype(int).values

def _club_sparse(d, n, min_frac):
    d = d.sort_index().copy()
    while len(d) > 2:
        share = d["cnt"] / n
        if share.min() >= min_frac: break
        j = int(share.values.argmin()); k = j + 1 if j == 0 else j - 1
        lo, hi = min(j, k), max(j, k)
        d.loc[d.index[lo], ["cnt", "bad"]] += d.loc[d.index[hi], ["cnt", "bad"]].values
        d.loc[d.index[lo], "hi"] = max(d.loc[d.index[lo], "hi"], d.loc[d.index[hi], "hi"])
        d.loc[d.index[lo], "lo"] = min(d.loc[d.index[lo], "lo"], d.loc[d.index[hi], "lo"])
        d = d.drop(d.index[hi]).reset_index(drop=True)
    return d

def _woe_iv(cnt, bad):
    good = cnt - bad; tb, tg = bad.sum(), good.sum()
    pb = (bad / tb).clip(lower=1e-6); pg = (good / tg).clip(lower=1e-6)
    return float(((pg - pb) * np.log(pg / pb)).sum())

_pdf = spark.table(CUST_TBL).where(_F.col("anchor_month") == PLOT_ANCHOR).select("target_fraud", *PLOT_FEATURES).toPandas()
_y = _pdf["target_fraud"].astype(int).values; _tot_bad = int(_y.sum()); _tot_good = len(_y) - _tot_bad
print("plotting anchor %s | customers=%d | frauds=%d" % (PLOT_ANCHOR, len(_pdf), _tot_bad))

def plot_sloping(feature):
    x = _pdf[feature].astype(float).values
    d = (pd.DataFrame({"b": _robust_bins(x), "x": x, "y": _y}).groupby("b")
         .agg(cnt=("y", "size"), bad=("y", "sum"), lo=("x", "min"), hi=("x", "max")).sort_index())
    d = _club_sparse(d, len(_pdf), MIN_BIN_FRAC)
    d = (d.iloc[::-1] if RISKY_ON_LEFT else d).reset_index(drop=True)
    d["event_rate_pct"] = 100.0 * d["bad"] / d["cnt"]
    cum_bad = d["bad"].cumsum(); cum_good = (d["cnt"] - d["bad"]).cumsum()
    d["pct_captured"] = 100.0 * cum_bad / max(_tot_bad, 1)
    ks = float((100.0 * cum_bad / max(_tot_bad, 1) - 100.0 * cum_good / max(_tot_good, 1)).abs().max())
    iv = _woe_iv(d["cnt"], d["bad"])
    d["er_lcb_pct"] = 100.0 * _wilson_lcb(d["bad"].values, d["cnt"].values)
    d["stable"] = (d["cnt"] >= MIN_STABLE_CNT) & (d["bad"] >= MIN_STABLE_BAD)
    labels = [("%.3g" % lo) if lo == hi else ("[%.3g, %.3g]" % (lo, hi)) for lo, hi in zip(d["lo"], d["hi"])]
    xs = range(len(d))
    fig, ax1 = plt.subplots(figsize=(11, 5))
    ax1.bar(xs, d["event_rate_pct"], color="#c97b84", alpha=0.85, label="Event Rate (%)")
    for i, v in enumerate(d["event_rate_pct"]): ax1.text(i, v, "%.2f" % v, ha="center", va="bottom", fontsize=8)
    ax1.set_ylabel("Event Rate (%)"); ax1.set_xticks(list(xs)); ax1.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    for i, st in enumerate(d["stable"].values):
        if not st: ax1.patches[i].set_hatch("///"); ax1.patches[i].set_edgecolor("#7a2a34")
    ax2 = ax1.twinx()
    ax2.plot(xs, d["pct_captured"], color="#a23b4e", marker="o", label="Cumulative % Fraud Captured")
    ax2.set_ylabel("Cumulative % Fraud Captured"); ax2.set_ylim(0, 105)
    ax1.set_title("%s @ %s   (IV=%.3f, KS=%.1f)   hatched = UNSTABLE (n<%d)\nEvent Rate (bars) & Cumulative %% Fraud Captured (line)"
                  % (feature, PLOT_ANCHOR, iv, ks, MIN_STABLE_CNT), fontsize=12)
    ax1.legend(loc="upper left"); ax2.legend(loc="lower right"); fig.tight_layout(); plt.show()
    _disp.display(pd.DataFrame({"bin": labels, "# frauds": d["bad"].astype(int).values,
                                "# customers": d["cnt"].astype(int).values,
                                "event_rate_%": d["event_rate_pct"].round(3).values,
                                "event_rate_LCB_%": d["er_lcb_pct"].round(2).values,
                                "cum_%_captured": d["pct_captured"].round(2).values,
                                "stable": d["stable"].values}))

for _f in PLOT_FEATURES:
    try: plot_sloping(_f)
    except Exception as _e: print("  %s: skipped (%s)" % (_f, _e))
